In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import configs
import ddpm_class
import FM_class
import evaluate
import plot_func


In [ ]:
cddpm = ddpm_class.cDDPM(save_int=80000, have_rho=False, cddpm_name="cddpm")
cddpm_have_rho = ddpm_class.cDDPM(save_int=80000, have_rho=True, cddpm_name="cddpm_rho")
tddpm = ddpm_class.tDDPM(save_int=80000, tddpm_name="tddpm")

tFM = FM_class.tFM(save_int=80000, tFM_name="tFM")
cFM = FM_class.cFM(save_int=80000, have_rho=False, cFM_name="cFM")
cFM_have_rho = FM_class.cFM(save_int=80000, have_rho=True, cFM_name="cFM_rho")

In [ ]:
cddpm.train(40000)

In [ ]:
cddpm_have_rho.train(40000)

In [ ]:
epoches = 80000
cFM.train(epoches)
cFM_have_rho.train(epoches)

In [ ]:
cFM.train( 80000)

In [ ]:
tddpm.load_ckpt('./saved_model/tddpm_it_80000.pth',)
cddpm.load_ckpt('./saved_model/cddpm_it_80000.pth')
cddpm_have_rho.load_ckpt('./saved_model/cddpm_rho_it_80000.pth')
tFM.load_ckpt('./saved_model/tFM_it_80000.pth',)
cFM.load_ckpt('./saved_model/cFM_it_80000.pth')
cFM_have_rho.load_ckpt('./saved_model/cFM_rho_it_80000.pth')

In [ ]:
from evaluate_nosave import eval_models, plot_models
# eval_models([tddpm, cddpm, cddpm_have_rho], 10, [False, False, True])
plot_models([tddpm, cddpm, cddpm_have_rho, tFM, cFM, cFM_have_rho], 10, [False, False, True, False, False, True], save=True)



In [ ]:
%matplotlib notebook
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


data = np.load('./output/test_record.npz')
MSE_data = data['MSE_record']
SWD_data = data['SWD_record']
# model_name = ['tddpm', 'cddpm', 'cddpm_have_rho', 'tFM', 'cFM', 'cFM_have_rho']
model_name = ['EI-FM', 'cFM', 'cFM-$\\gamma$']

rho = None
mu1 = 0
mu2 = 2.5

def plot_3D(input_data, model_name, title, ylabel, rho, save_name=None):
    mu1=np.linspace(-2.5,2.5,41)
    mu2=np.linspace(-2.5,2.5,41)
    rho_axis = np.linspace(-1,1,21)
    x, y = np.meshgrid(mu1, mu2)
    plotdata = input_data[:, np.where(np.abs(rho_axis-rho)<1e-5)[0][0], :,:]
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    for i in range(len(model_name)):
        ax.plot_surface(x, y, plotdata[i], color = ['r','g','b'][i], label = model_name[i])
    plt.xlabel(r'$\mu_1$')
    plt.ylabel(r'$\mu_2$')
    ax.set_zlabel("SWD", rotation=90)
    ax.set_zlim([0,0.6])
    plt.legend()
    if save_name is not None:
        # plt.tight_layout()
        plt.savefig('./fig/' + save_name + '.png', bbox_inches='tight', dpi=300,pad_inches=0.21)
    # plt.title(title)
    plt.show()

def plot_data(input_data, model_name, title, ylabel, z, save_name = None):
    rho = z[0]
    mu1 = z[1]
    mu2 = z[2]
    rho_axis = np.linspace(-1,1,21)
    mu_axis = np.linspace(-2.5,2.5,41)
    
    if rho is None and mu1 is not None and mu2 is not None:
        x = np.linspace(-1,1,21)
        plotdata = input_data[:,:,np.where(mu_axis==mu1)[0][0],np.where(mu_axis==mu2)[0][0]]
    elif rho is not None and mu1 is None and mu2 is not None:
        x = np.linspace(-2.5,2.5,41)
        plotdata = input_data[:,np.where(rho_axis==rho)[0][0],:,np.where(mu_axis==mu2)[0][0]]
    elif rho is not None and mu1 is not None and mu2 is None:
        x = np.linspace(-2.5,2.5,41)
        plotdata = input_data[:,np.where(rho_axis==rho)[0][0],np.where(mu_axis==mu1)[0][0],:]
    else:
        raise ValueError('only 1 None')

    for i in range(len(model_name)):
        plt.plot(x, plotdata[i], label=model_name[i])
    # plt.axvspan(-0.75, -0.25, color='grey', alpha=0.5)
    # plt.axvspan(0.25, 0.75, color='grey', alpha=0.5)
    plt.grid()
    plt.legend()
    if rho is None:
        plt.xlabel(r'$\rho$')
    elif mu1 is None:
        plt.xlabel(r'$\mu_1$')
    elif mu2 is None:
        plt.xlabel(r'$\mu_2$')


    plt.ylabel(ylabel)
    plt.title(title)
    if save_name is not None:
        plt.savefig('./fig/' + save_name + '.png', bbox_inches='tight', dpi=300)
    plt.show()


z = (None, 1.5, -1.5)
# plot_data(MSE_data, model_name, title='MSE_record', ylabel="MSE", z=z, save_name='MSE_record')
# plot_data(SWD_data, model_name, title='SWD_record', ylabel="SWD", z=z, save_name='SWD_record')

# plot_3D(MSE_data, model_name, title='MSE_record', ylabel="MSE", rho=0, save_name='MSE_record')




In [ ]:
rho_list=[-0.9,-0.5,0.0,0.5,0.9]
for rho in rho_list:
    rhoname = str(rho).replace('-','n').replace('.','p')
    plot_3D(SWD_data, model_name, title='SWD_record_'+str(rho), ylabel="SWD", rho=rho, save_name='SWD_record'+rhoname)